In [1]:
import pandas as pd
import numpy as np
import nltk

#Biblioteca de PLN mais moderna e rápida
import spacy

#Divide seus dados em conjunto de treino
from sklearn.model_selection import train_test_split


#algoritmo de classificação que cria muitas árvores de decisão e combina os votos delas.
from sklearn.ensemble import RandomForestClassifier

#algoritmo de classificação que busca a "fronteira" que melhor separa as classes.
from sklearn.svm import SVC

#algoritmo de classificação regressão logistica
from sklearn.linear_model import LogisticRegression

#algoritmo de classificação 
from sklearn.naive_bayes import MultinomialNB


#Gera um relatório de avaliação com precisão, recall, F1-score e acurácia para cada classe.
from sklearn.metrics import classification_report

#Salva e carrega o modelo treinado em arquivo
import joblib


# Downloads essenciais para viabilizar as ferramentas de NLP do NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

# Carregamento do modelo de português do spaCy (exige instalação prévia: python -m spacy download pt_core_news_sm)
nlp = spacy.load("pt_core_news_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pedro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pedro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\pedro\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pedro\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [ ]:
from IPython.display import display
from pathlib import Path
import os
import pandas as pd
from sklearn.model_selection import train_test_split

def carregar_e_dividir_dados(caminho_base_dataset):
    """
    Lê os arquivos .txt locais, estrutura em um DataFrame e divide em Treino, Validação e Teste.
    """
    dados = []
    
    # 1. Mapeamento das categorias exigidas pelo projeto
    categorias = [
        "1_Ensino_Fundamental_I", 
        "2_Ensino_Fundamental_II", 
        "3_Ensino_Medio", 
        "4_Ensino_Superior"
    ]
    
    # 2. Leitura e Estruturação dos Dados Locais
    for categoria in categorias:
        caminho_pasta = os.path.join(caminho_base_dataset, categoria) #caminho_pasta = dataset/Ensino_Fundamental_I
        
        # Verifica se a pasta existe antes de iterar
        if os.path.exists(caminho_pasta):
            for nome_arquivo in os.listdir(caminho_pasta):
                if nome_arquivo.endswith('.txt'):
                    caminho_arquivo = os.path.join(caminho_pasta, nome_arquivo) #caminho arquivo = dataset/Ensino_Fundamental_I/arquivo1.txt
                    
                    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                        texto = f.read().strip()
                        
                        if texto:
                            dados.append({"texto": texto, "complexidade": categoria})
    
    # Converte a lista de dicionários para um DataFrame do Pandas
    df = pd.DataFrame(dados)

    if df.empty:
        raise ValueError("Nenhum dado foi carregado. Verifique os caminhos dos arquivos.")


    # 3. Divisão dos Dados (Treino, Validação e Teste) --> ordem: (maior,menor)
    # Primeiro, separamos o Teste (ex: 15% dos dados) do resto (Treino + Validação)
    # O parâmetro stratify garante que a proporção das 4 categorias se mantenha igual
    df_resto, df_teste = train_test_split(
        df, 
        test_size=0.15, 
        stratify=df['complexidade'], 
        random_state=42                # fixa a semente do sorteio, então a divisão é sempre a mesma toda vez que você rodar. O número 42 é só uma convenção, qualquer inteiro serve.
    )
    
    # Agora, dividimos o "resto" em Treino (ex: 70% do total) e Validação (ex: 15% do total)
    # Como tiramos 15% antes, restaram 85%. Para tirar ~15% do total a partir dos 85%, usamos test_size=0.176 (0.15/0.85)
    df_treino, df_validacao = train_test_split(
        df_resto, 
        test_size=0.1764, 
        stratify=df_resto['complexidade'], 
        random_state=42
    )
    
    print(f"Dados carregados com sucesso!")
    print(f"Total: {len(df)} instâncias.")
    print(f"Treino: {len(df_treino)} | Validação: {len(df_validacao)} | Teste: {len(df_teste)}")
    
    return df_treino, df_validacao, df_teste





# Caminho da pasta principal
caminho_base_dataset = Path(r"c:\Users\pedro\Downloads\Prova de seleção\dataset_caed")

df_treino, df_validacao, df_teste = carregar_e_dividir_dados(caminho_base_dataset)

display(df_treino, df_validacao, df_teste)


Dados carregados com sucesso!
Total: 35597 sentenças.
Treino: 24919 | Validação: 5338 | Teste: 5340


,texto,complexidade
28165,"Agora, basta-nos substituir os valores na equa...",4_Ensino_Superior
33255,Logística/Sistemas de informação/Indicadores d...,4_Ensino_Superior
14212,"Ainda considerando o polinômio formula_38, vam...",3_Ensino_Medio
30354,Logística/Sistemas de distribuição/Modelos de ...,4_Ensino_Superior
21391,Apesar do hemidesmossoma ter aparência e funçõ...,3_Ensino_Medio
...,...,...
25577,Análise real/Unicidade dos números reais\nExis...,4_Ensino_Superior
4157,As brincadeiras ajudam a passar o tempo e dão ...,1_Ensino_Fundamental_I
12588,Química inorgânica/Introdução/Íons e a regra d...,3_Ensino_Medio
20954,A poluição é a liberação de substâncias químic...,3_Ensino_Medio


,texto,complexidade
14610,formula_50\nAgora podemos generalizar dizendo ...,3_Ensino_Medio
7221,Sociologia. A sociologia de Max Weber influenc...,2_Ensino_Fundamental_II
15942,"Posteriormente, a companhia se mudou para as c...",3_Ensino_Medio
12831,Introdução à Química/Cadeias carbônicas\nDevid...,3_Ensino_Medio
28762,"Se existem, o supremo e ínfimo de um conjunto ...",4_Ensino_Superior
...,...,...
4626,"10, 9, 8, 7, 6, 5, 4, 3, 2, 1... LANÇAR!\nNão ...",1_Ensino_Fundamental_I
5489,"Para os bolsos de seus senhores, é claro. Você...",1_Ensino_Fundamental_I
18013,"Sigmund Freud/Teoria Freudiana\nSegundo Freud,...",3_Ensino_Medio
3089,Acertou em cheio quem se lembrou do Sítio do P...,1_Ensino_Fundamental_I


,texto,complexidade
33569,(2006) . LAC é um algoritmo não paramétrico ba...,4_Ensino_Superior
21136,A osmose é o nome dado ao movimento da água en...,3_Ensino_Medio
11525,"Política. No regime democrático, a existência ...",2_Ensino_Fundamental_II
10107,﻿ Com olhos vivos e sorriso sempre estampado n...,2_Ensino_Fundamental_II
16339,"Por este motivo, a igreja mudou sua denominaçã...",3_Ensino_Medio
...,...,...
31548,Logística/Manutenção de instalações/Estrutura ...,4_Ensino_Superior
18258,Frize-se que a nossa gramática tradicional é u...,3_Ensino_Medio
10027,"﻿Antes do Cinema, veio a fotografia. Sem ela, ...",2_Ensino_Fundamental_II
5685,Mas uma coisa você tem mesmo que agradecer aos...,1_Ensino_Fundamental_I
